# Day 13 · Zillacode prep 2/4: dates, aggregations, pivots and arrays

Yesterday was about single rows. Today is about time and groups:
1. Dates and timestamps (in Zillacode they always arrive as text)
2. Aggregations: `groupBy(...).agg(...)`
3. Aggregate first, then join: the fan-out trap
4. Pivot: rows into columns
5. Arrays: `split`, `explode` and `collect_list`

In Zillacode's topic list this is *Datetime Operations*, *Aggregate Functions* (the most common topic after joins) and *Pivot Function*. The setup cell is the same as yesterday's.

## How to work
* Run the **Setup** cell first, then the cell that prints the tables.
* **Lessons** explain one tool at a time. Read them, run every example cell (the output is already shown under it), then change the example and run it again until you can predict the result.
* **Exercises** come right after each lesson and are in Zillacode's format: complete `etl(...)` in the cell under the task so that it **returns** a DataFrame, then run the cell. The last line, `etl(...).show()`, prints your result. It must match the expected output: same columns, same rows, same order. (Zillacode itself ignores the order of rows and columns; the expected outputs here are sorted only so you can compare them by eye.)
* The answers are collapsed at the very end of the notebook. Open one only after you have tried, then close it and write the solution again yourself.
* Today: about 1 hour of lessons and 2 hours of exercises (17 quick ones, then 3 mixed problems).

**Setup (once):** Java 17 or 21 and Python 3.10 or newer. In a new folder: `python3.12 -m venv .venv`, `source .venv/bin/activate`, `pip install pyspark==4.2.0 jupyterlab`, then `jupyter lab` and open this notebook.

## The data
A small online shop. The setup cell creates every table from rows you can read, and the cell after it prints them.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date*
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)
* **reviews**: *review_id*, *product_id*, *customer_id*, *review_date* (text like `15/01/2024`), *rating* (1 to 5), *comment* (messy text), *tags* (text like `fast delivery;good price`)
* **signups_raw**: sign-ups exactly as a web form sent them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text in three formats), *amount* (text), *prefs* (JSON text)
* **customers_new**: the customers table one week later
* **promotions**: *promo_id*, *category*, *start_date*, *end_date* (both days included), *discount_pct*
* **events**: website visits: *customer_id*, *event_time* (a timestamp), *page*
* **referrals**: *referrer_id* (the customer who invited someone), *referred_id* (the customer who joined), *referral_date*

The data has the traps that Zillacode's hidden tests like to use: order 1011 is a guest checkout (no *customer_id*); customer 6 has no email and no city; customer 7 has never ordered; product 107 has never been sold; order 1016 is for product 108, which is not in *products*; review 6 has no tags; two events were loaded twice; referral 5 points to customer 9, who doesn't exist.

## Setup
Run this cell once. It starts Spark and creates the tables.

In [ ]:
# Setup: starts Spark and creates the tables. Run it once.
import logging, os, sys
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F, Window as W   # the same names as in Zillacode's editor

spark = (SparkSession.builder
         .master("local[4]")
         .appName("zillacode-prep-13")
         .config("spark.sql.ansi.enabled", "false")        # behave like Zillacode (Spark 3): bad text gives null, not an error
         .config("spark.sql.shuffle.partitions", "4")      # small data: 4 shuffle partitions are enough
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")               # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers = spark.createDataFrame([
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
], "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE")

products = spark.createDataFrame([
    # product_id, product_name, category, price (dollars for one item)
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
], "product_id INT, product_name STRING, category STRING, price DOUBLE")

orders = spark.createDataFrame([
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
], "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING")

reviews = spark.createDataFrame([
    # review_id, product_id, customer_id, review_date (text), rating, comment, tags (text)
    (1, 101, 1, "15/01/2024", 5, "  Great laptop, very FAST!  ",    "fast delivery;good price"),
    (2, 105, 1, "20/01/2024", 4, "Nice story.",                     "good price"),
    (3, 102, 2, "02/02/2024", 3, "Sound is ok, delivery was slow",  "slow delivery"),
    (4, 103, 3, "10/02/2024", 1, "Broke after one week!!",          "broken;slow delivery"),
    (5, 101, 5, "09/03/2024", 5, "Fast and powerful",               "fast delivery"),
    (6, 105, 3, "12/03/2024", 2, "Too long, boring",                None),
    (7, 106, 2, "25/03/2024", 4, "Useful recipes",                  "good price;gift"),
    (8, 103, 6, "30/03/2024", 4, "Good coffee, slow start",         "gift"),
], "review_id INT, product_id INT, customer_id INT, review_date STRING, rating INT, comment STRING, tags STRING")

signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text, three formats), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

customers_new = spark.createDataFrame([
    # the customers table one week later: 2 moved, 6 has an email now, 7 is gone, 8 is new
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Munich",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     "farid@mail.az",    None,       date(2024, 3, 10)),
    (8, "Hana Sato",        "hana@mail.jp",     "Osaka",    date(2024, 4, 2)),
], customers.schema)

promotions = spark.createDataFrame([
    # promo_id, category, start_date, end_date (both days included), discount_pct
    ("WINTER",   "Electronics", date(2024, 1, 1),  date(2024, 1, 31), 10),
    ("BOOKWEEK", "Books",       date(2024, 3, 1),  date(2024, 3, 10), 20),
    ("SPRING",   "Kitchen",     date(2024, 3, 1),  date(2024, 3, 31), 15),
    ("FLASH",    "Electronics", date(2024, 3, 15), date(2024, 3, 20), 5),
], "promo_id STRING, category STRING, start_date DATE, end_date DATE, discount_pct INT")

events = spark.createDataFrame([
    # customer_id, event_time, page (two rows were loaded twice)
    (1, "2024-03-01 09:00:00", "home"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 09:10:00", "product"),
    (1, "2024-03-01 11:00:00", "home"),
    (1, "2024-03-02 10:00:00", "cart"),
    (1, "2024-03-03 10:30:00", "home"),
    (1, "2024-03-05 08:00:00", "home"),
    (2, "2024-03-01 20:00:00", "home"),
    (2, "2024-03-01 20:20:00", "product"),
    (2, "2024-03-01 20:45:00", "cart"),
    (2, "2024-03-04 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (2, "2024-03-05 18:00:00", "home"),
    (3, "2024-03-02 12:00:00", "product"),
    (3, "2024-03-03 12:00:00", "product"),
    (3, "2024-03-04 12:00:00", "cart"),
], "customer_id INT, event_time STRING, page STRING").withColumn("event_time", F.to_timestamp("event_time"))

referrals = spark.createDataFrame([
    # referrer_id (the customer who invited), referred_id (the customer who joined), referral_date
    (1, 4, date(2024, 2, 10)),
    (4, 5, date(2024, 2, 25)),
    (1, 6, date(2024, 3, 8)),
    (2, 7, date(2024, 3, 12)),
    (2, 9, date(2024, 3, 30)),
], "referrer_id INT, referred_id INT, referral_date DATE")

print("Spark", spark.version, "is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, "
      "promotions, events, referrals")

Spark 4.2.0 is ready. Tables: customers, products, orders, reviews, signups_raw, customers_new, promotions, events, referrals


Print every table, so you can check results by eye:

In [ ]:
for name in ["customers", "products", "orders", "reviews", "signups_raw", "customers_new", "promotions", "events", "referrals"]:
    print(name)
    globals()[name].show(truncate=False)

customers
+-----------+----------------+----------------+--------+-----------+
|customer_id|name            |email           |city    |signup_date|
+-----------+----------------+----------------+--------+-----------+
|1          |Aziz Karimov    |aziz@mail.uz    |Tashkent|2024-01-05 |
|2          |Bella Schmidt   |bella@web.de    |Berlin  |2024-01-20 |
|3          |Carlos Ruiz     |carlos@correo.es|Madrid  |2024-02-02 |
|4          |Dilnoza Rahimova|dilnoza@mail.uz |Tashkent|2024-02-14 |
|5          |Emma Wilson     |emma@post.co.uk |London  |2024-03-01 |
|6          |Farid Aliyev    |NULL            |NULL    |2024-03-10 |
|7          |Grace Lee       |grace@web.de    |Berlin  |2024-03-15 |
+-----------+----------------+----------------+--------+-----------+

products
+----------+------------+-----------+------+
|product_id|product_name|category   |price |
+----------+------------+-----------+------+
|101       |Laptop      |Electronics|1200.0|
|102       |Headphones  |Electronics|150.

## Lesson 1 · Dates and timestamps

Spark has a `date` type (`2024-03-02`), a `timestamp` type (`2024-03-02 09:10:00`), and then there's text that only *looks* like a date. **In Zillacode every date arrives as text** (Day 12, Lesson 0), so the first step is usually a conversion:

| Function | What it does |
|---|---|
| `F.to_date(col)` | text in the form `yyyy-MM-dd` to a date |
| `F.to_date(col, "dd/MM/yyyy")` | text in another form to a date: the pattern describes the **text** |
| `F.to_timestamp(col, "yyyy-MM-dd HH:mm:ss")` | text to a timestamp |
| `F.date_format(col, "yyyy-MM")` | a date or timestamp to text, in any pattern |

**Pattern letters** ([full list](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html)). Case matters:

| Letters | Meaning | Example |
|---|---|---|
| `yyyy` | year | 2024 |
| `MM`, `M` | month with / without a leading zero | 03, 3 |
| `MMM`, `MMMM` | month name | Mar, March |
| `dd`, `d` | day of the month | 05, 5 |
| `HH` | hour, 0 to 23 | 18 |
| `mm` | **minutes** (not months!) | 45 |
| `ss` | seconds | 00 |
| `E`, `EEEE` | weekday name | Sat, Saturday |

`MM` vs `mm` is the classic bug: `"yyyy-mm-dd"` reads the month digits as minutes. Text that doesn't fit the pattern becomes **null** (on Zillacode, and here with ANSI off). An end date of `"ongoing"`, for example, turns into null, and you can then treat that null as "still running".

**Parts and date math**

| Function | Result |
|---|---|
| `F.year`, `F.month`, `F.dayofmonth`, `F.quarter`, `F.weekofyear` | parts as numbers |
| `F.dayofweek(d)` | **1 = Sunday**, 2 = Monday, ..., 7 = Saturday. Weekend: `F.dayofweek(d).isin(1, 7)` |
| `F.hour`, `F.minute`, `F.second` | parts of a timestamp |
| `F.datediff(end, start)` | whole days from start to end (end minus start) |
| `F.months_between(end, start)` | months, with a fraction |
| `F.date_add(d, n)`, `F.date_sub(d, n)`, `F.add_months(d, n)` | move a date by days or months |
| `F.last_day(d)`, `F.trunc(d, "month")` | the last / first day of the month (`trunc` also takes `"year"`) |
| `F.date_trunc("hour", ts)` | a timestamp cut down to the hour (or `"day"`, `"month"`, ...) |
| `F.unix_timestamp(ts)` | seconds since 1970. Subtract two of them for a duration in seconds; divide by 60 for minutes. |
| `F.current_date()` | today: avoid it on Zillacode, where problems state their own "today" ("assume the current year is 2024") |

A date column can be compared with a text literal: `F.col("order_date") >= "2024-03-01"` works, and `between("2024-03-01", "2024-03-31")` includes both ends.

On Zillacode: #1 (released in the last 5 years), #20 (a project's duration in days), #22 (minutes watched while a subscription was active, with `"ongoing"` end dates), #31 (weekend orders from `MM/DD/YYYY` text), and #8 and #35 (one row per date).

In [ ]:
# Text to date: the pattern must describe the text, or you get null
reviews.select(
    "review_date",
    F.to_date("review_date", "dd/MM/yyyy").alias("as_date"),
    F.to_date("review_date").alias("wrong_pattern"),
).show(4)

+-----------+----------+-------------+
|review_date|   as_date|wrong_pattern|
+-----------+----------+-------------+
| 15/01/2024|2024-01-15|         NULL|
| 20/01/2024|2024-01-20|         NULL|
| 02/02/2024|2024-02-02|         NULL|
| 10/02/2024|2024-02-10|         NULL|
+-----------+----------+-------------+
only showing top 4 rows


In [ ]:
day = F.to_date("review_date", "dd/MM/yyyy")
reviews.select(
    "review_id", day.alias("day"),
    F.year(day).alias("year"), F.month(day).alias("month"), F.dayofweek(day).alias("dayofweek"),
    F.date_format(day, "EEEE").alias("weekday"),
    F.dayofweek(day).isin(1, 7).alias("weekend"),
    F.date_format(day, "yyyy-MM").alias("year_month"),
).show()

+---------+----------+----+-----+---------+--------+-------+----------+
|review_id|       day|year|month|dayofweek| weekday|weekend|year_month|
+---------+----------+----+-----+---------+--------+-------+----------+
|        1|2024-01-15|2024|    1|        2|  Monday|  false|   2024-01|
|        2|2024-01-20|2024|    1|        7|Saturday|   true|   2024-01|
|        3|2024-02-02|2024|    2|        6|  Friday|  false|   2024-02|
|        4|2024-02-10|2024|    2|        7|Saturday|   true|   2024-02|
|        5|2024-03-09|2024|    3|        7|Saturday|   true|   2024-03|
|        6|2024-03-12|2024|    3|        3| Tuesday|  false|   2024-03|
|        7|2024-03-25|2024|    3|        2|  Monday|  false|   2024-03|
|        8|2024-03-30|2024|    3|        7|Saturday|   true|   2024-03|
+---------+----------+----+-----+---------+--------+-------+----------+


In [ ]:
orders.select(
    "order_id", "order_date",
    F.datediff(F.to_date(F.lit("2024-04-01")), "order_date").alias("days_until_april"),
    F.date_add("order_date", 30).alias("plus_30_days"),
    F.add_months("order_date", 1).alias("plus_1_month"),
    F.trunc("order_date", "month").alias("month_start"),
    F.last_day("order_date").alias("month_end"),
).show(5)

+--------+----------+----------------+------------+------------+-----------+----------+
|order_id|order_date|days_until_april|plus_30_days|plus_1_month|month_start| month_end|
+--------+----------+----------------+------------+------------+-----------+----------+
|    1001|2024-01-10|              82|  2024-02-09|  2024-02-10| 2024-01-01|2024-01-31|
|    1002|2024-01-10|              82|  2024-02-09|  2024-02-10| 2024-01-01|2024-01-31|
|    1003|2024-01-25|              67|  2024-02-24|  2024-02-25| 2024-01-01|2024-01-31|
|    1004|2024-02-03|              58|  2024-03-04|  2024-03-03| 2024-02-01|2024-02-29|
|    1005|2024-02-05|              56|  2024-03-06|  2024-03-05| 2024-02-01|2024-02-29|
+--------+----------+----------------+------------+------------+-----------+----------+
only showing top 5 rows


In [ ]:
# Durations between timestamps: unix_timestamp gives seconds
start = F.to_timestamp(F.lit("2024-03-01 20:00:00"))
events.filter(F.col("customer_id") == 2).select(
    "event_time",
    F.hour("event_time").alias("hour"),
    F.date_trunc("day", "event_time").alias("day_start"),
    ((F.unix_timestamp("event_time") - F.unix_timestamp(start)) / 60).alias("minutes_after_start"),
).show()

+-------------------+----+-------------------+-------------------+
|         event_time|hour|          day_start|minutes_after_start|
+-------------------+----+-------------------+-------------------+
|2024-03-01 20:00:00|  20|2024-03-01 00:00:00|                0.0|
|2024-03-01 20:20:00|  20|2024-03-01 00:00:00|               20.0|
|2024-03-01 20:45:00|  20|2024-03-01 00:00:00|               45.0|
|2024-03-04 18:00:00|  18|2024-03-04 00:00:00|             4200.0|
|2024-03-05 18:00:00|  18|2024-03-05 00:00:00|             5640.0|
|2024-03-05 18:00:00|  18|2024-03-05 00:00:00|             5640.0|
+-------------------+----+-------------------+-------------------+


### Exercises

**Q1 · `etl(customers)`** How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered, sorted by signup_date.

Expected output:

```
+----------------+-----------+---------------+
|            name|signup_date|days_registered|
+----------------+-----------+---------------+
|    Aziz Karimov| 2024-01-05|             87|
|   Bella Schmidt| 2024-01-20|             72|
|     Carlos Ruiz| 2024-02-02|             59|
|Dilnoza Rahimova| 2024-02-14|             47|
|     Emma Wilson| 2024-03-01|             31|
|    Farid Aliyev| 2024-03-10|             22|
|       Grace Lee| 2024-03-15|             17|
+----------------+-----------+---------------+
```

In [ ]:
def etl(customers):
    # Write code here: return a DataFrame
    return customers

etl(customers).show()

**Q2 · `etl(orders)`** For the orders placed in January and February, show order_id, order_date, the last day of that month (`month_end`) and the days left until then (`days_left`), sorted by order_id.

Expected output:

```
+--------+----------+----------+---------+
|order_id|order_date| month_end|days_left|
+--------+----------+----------+---------+
|    1001|2024-01-10|2024-01-31|       21|
|    1002|2024-01-10|2024-01-31|       21|
|    1003|2024-01-25|2024-01-31|        6|
|    1004|2024-02-03|2024-02-29|       26|
|    1005|2024-02-05|2024-02-29|       24|
|    1006|2024-02-15|2024-02-29|       14|
|    1007|2024-02-20|2024-02-29|        9|
+--------+----------+----------+---------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q3 · `etl(signups_raw)`** The sign-up dates come in three formats (`2024-01-05`, `20/01/2024`, `2024/02/02`). Turn them into one date column `signup_day`. Show signup_id, signup_date and signup_day, sorted by signup_id.

Expected output:

```
+---------+-----------+----------+
|signup_id|signup_date|signup_day|
+---------+-----------+----------+
|        1| 2024-01-05|2024-01-05|
|        2| 20/01/2024|2024-01-20|
|        3| 2024/02/02|2024-02-02|
|        4| 14/02/2024|2024-02-14|
|        5| 2024-03-01|2024-03-01|
|        6| 2024-03-02|2024-03-02|
+---------+-----------+----------+
```

In [ ]:
def etl(signups_raw):
    # Write code here: return a DataFrame
    return signups_raw

etl(signups_raw).show()

**Q4 · `etl(promotions)`** How long does each promotion run, counting both the first and the last day? Show promo_id, start_date, end_date and days, sorted by start_date, then promo_id.

Expected output:

```
+--------+----------+----------+----+
|promo_id|start_date|  end_date|days|
+--------+----------+----------+----+
|  WINTER|2024-01-01|2024-01-31|  31|
|BOOKWEEK|2024-03-01|2024-03-10|  10|
|  SPRING|2024-03-01|2024-03-31|  31|
|   FLASH|2024-03-15|2024-03-20|   6|
+--------+----------+----------+----+
```

In [ ]:
def etl(promotions):
    # Write code here: return a DataFrame
    return promotions

etl(promotions).show()

**Q5 · `etl(events)`** Show the evening events (18:00 or later), duplicates removed: customer_id, event_time and page, sorted by event_time.

Expected output:

```
+-----------+-------------------+-------+
|customer_id|         event_time|   page|
+-----------+-------------------+-------+
|          2|2024-03-01 20:00:00|   home|
|          2|2024-03-01 20:20:00|product|
|          2|2024-03-01 20:45:00|   cart|
|          2|2024-03-04 18:00:00|   home|
|          2|2024-03-05 18:00:00|   home|
+-----------+-------------------+-------+
```

In [ ]:
def etl(events):
    # Write code here: return a DataFrame
    return events

etl(events).show()

## Lesson 2 · Aggregations: `groupBy(...).agg(...)`

`groupBy` collects the rows that have the same key, and `agg` computes values for each group. The result has **one row per group**: the key columns plus one column per aggregate.

```python
orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"),
    F.sum("quantity").alias("items"),
)
```

Always `alias` an aggregate; otherwise the column is called something like `sum(quantity)`.

| Function | Gives |
|---|---|
| `F.count("*")` | the number of rows in the group |
| `F.count("col")` | the number of rows where col is **not null** |
| `F.countDistinct("col")` (also `F.count_distinct`) | the number of different non-null values |
| `F.sum`, `F.avg` (also `F.mean`), `F.min`, `F.max` | they **ignore nulls**: the avg of 1, null, 3 is 2 |
| `F.round(F.avg("x"), 2)` | round the result, not the inputs |
| `F.collect_list("x")`, `F.collect_set("x")` | all values / the different values, as an array (Lesson 5) |
| `F.first("x")` | some value of the group; which one isn't defined without an order |
| `F.min_by("x", "y")`, `F.max_by("x", "y")` | x from the row with the smallest / largest y |
| `F.stddev`, `F.variance` | spread (the sample versions) |

Ways to group:
* `groupBy("a", "b")` makes one group per (a, b) combination.
* `groupBy(F.month("order_date").alias("month"))` groups by an expression.
* `df.agg(...)` without `groupBy` gives **one row** for the whole table.
* `groupBy("x").count()` is a shortcut that adds a column named `count`.
* A null key forms its own group: the guest order shows up with customer_id null.

**Filter on a group (SQL `HAVING`).** `filter` after `agg`, on the alias: `.agg(F.count("*").alias("n")).filter(F.col("n") >= 3)`.

**Conditional aggregation** counts or sums only some rows of each group: `F.sum(F.when(cond, 1).otherwise(0))` or `F.count(F.when(cond, True))`. The second works because `count` skips the nulls that a `when` without `otherwise` produces. The same idea gives conditional sums: `F.sum(F.when(F.col("status") == "delivered", F.col("quantity")))`.

**One number from the whole table.** `total = orders.agg(F.sum("quantity")).first()[0]` gives a plain Python value that you can use in `F.lit(total)` or in a filter. Zillacode's #32 compares each ride with the overall average this way. You can stay inside Spark instead with a `crossJoin` of the one-row aggregate (Day 14), or with a window over the whole table (Day 14).

On Zillacode, aggregations are in #5, #8, #19, #20, #23, #25, #35, #36, #39 and #44. [GroupedData docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.agg.html).

In [ ]:
products.groupBy("category").agg(
    F.count("*").alias("products"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
).orderBy("category").show()

+-----------+--------+---------+---------+---------+
|   category|products|avg_price|min_price|max_price|
+-----------+--------+---------+---------+---------+
|      Books|       2|     20.0|     15.0|     25.0|
|Electronics|       2|    675.0|    150.0|   1200.0|
|       Home|       1|     40.0|     40.0|     40.0|
|    Kitchen|       2|     72.5|     60.0|     85.0|
+-----------+--------+---------+---------+---------+


In [ ]:
# count(*) vs count(col) vs countDistinct; the null key is its own group
orders.groupBy("customer_id").agg(
    F.count("*").alias("rows"),
    F.countDistinct("product_id").alias("different_products"),
    F.sum("quantity").alias("items"),
).orderBy("customer_id").show()
customers.agg(F.count("*").alias("rows"), F.count("email").alias("with_email"), F.countDistinct("city").alias("cities")).show()

+-----------+----+------------------+-----+
|customer_id|rows|different_products|items|
+-----------+----+------------------+-----+
|       NULL|   1|                 1|    1|
|          1|   4|                 4|    6|
|          2|   3|                 3|    4|
|          3|   3|                 3|    6|
|          4|   2|                 2|    4|
|          5|   2|                 2|    6|
|          6|   1|                 1|    2|
+-----------+----+------------------+-----+

+----+----------+------+
|rows|with_email|cities|
+----+----------+------+
|   7|         6|     4|
+----+----------+------+


In [ ]:
# HAVING: keep only the big groups
orders.groupBy("status").agg(F.count("*").alias("orders"), F.sum("quantity").alias("items")).filter(F.col("orders") >= 4).orderBy("status").show()
# Conditional aggregation: count some rows of each group
orders.groupBy(F.month("order_date").alias("month")).agg(
    F.count("*").alias("orders"),
    F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
    F.count(F.when(F.col("quantity") >= 2, True)).alias("multi_item"),
    F.sum(F.when(F.col("status") == "delivered", F.col("quantity"))).alias("delivered_items"),
).orderBy("month").show()

+---------+------+-----+
|   status|orders|items|
+---------+------+-----+
|delivered|    10|   20|
|  shipped|     4|    6|
+---------+------+-----+

+-----+------+---------+----------+---------------+
|month|orders|delivered|multi_item|delivered_items|
+-----+------+---------+----------+---------------+
|    1|     3|        3|         2|              6|
|    2|     4|        2|         1|              2|
|    3|     9|        5|         4|             12|
+-----+------+---------+----------+---------------+


In [ ]:
# One value from the whole table, used in a filter
avg_price = products.agg(F.avg("price")).first()[0]
print("average price:", avg_price)
products.filter(F.col("price") > avg_price).select("product_name", "price").show()

average price: 225.0
+------------+------+
|product_name| price|
+------------+------+
|      Laptop|1200.0|
+------------+------+


### Exercises

**Q6 · `etl(products)`** For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by category.

Expected output:

```
+-----------+--------+---------+---------+
|   category|products|avg_price|max_price|
+-----------+--------+---------+---------+
|      Books|       2|     20.0|     25.0|
|Electronics|       2|    675.0|   1200.0|
|       Home|       1|     40.0|     40.0|
|    Kitchen|       2|     72.5|     85.0|
+-----------+--------+---------+---------+
```

In [ ]:
def etl(products):
    # Write code here: return a DataFrame
    return products

etl(products).show()

**Q7 · `etl(orders)`** Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.

Expected output:

```
+-----------+------+
|customer_id|orders|
+-----------+------+
|          1|     4|
|          2|     3|
|          3|     3|
+-----------+------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q8 · `etl(orders)`** How many different customers ordered in each month? Show month (1 to 3) and customers. The guest order doesn't count.

Expected output:

```
+-----+---------+
|month|customers|
+-----+---------+
|    1|        2|
|    2|        4|
|    3|        6|
+-----+---------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q9 · `etl(orders)`** For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show customer_id, delivered and cancelled, sorted by customer_id. Leave out the guest order.

Expected output:

```
+-----------+---------+---------+
|customer_id|delivered|cancelled|
+-----------+---------+---------+
|          1|        3|        0|
|          2|        3|        0|
|          3|        2|        1|
|          4|        0|        1|
|          5|        1|        0|
|          6|        0|        0|
+-----------+---------+---------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q10 · `etl(orders)`** Which customers order more items per order, on average, than the average order? Show customer_id and avg_quantity (2 decimals), sorted by customer_id. The guest isn't a customer, but the guest order counts in the overall average.

Expected output:

```
+-----------+------------+
|customer_id|avg_quantity|
+-----------+------------+
|          3|         2.0|
|          4|         2.0|
|          5|         3.0|
|          6|         2.0|
+-----------+------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q11 · `etl(orders, products)`** What share of all items ordered does each category have? Show category, items and share_pct (percent, 1 decimal), biggest share first. Use an inner join: order 1016, whose product is unknown, doesn't count anywhere.

Expected output:

```
+-----------+-----+---------+
|   category|items|share_pct|
+-----------+-----+---------+
|      Books|   15|     53.6|
|    Kitchen|    7|     25.0|
|Electronics|    6|     21.4|
+-----------+-----+---------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Lesson 3 · Aggregate first, then join: the fan-out trap

The Laptop has 3 orders and 2 reviews. Join products to orders and then to reviews, and the Laptop gets 3 × 2 = **6 rows**: every order repeats once per review. A `sum` or `count` over those rows counts everything several times. Nothing fails; the numbers are just wrong.

**The fix, used by many Zillacode problems (#10, #20, #23):**
1. Aggregate each detail table by the key **first**, so you have one row per key: `items per product`, `reviews per product`.
2. Join these small results to the main table with **left** joins, so keys without details stay in the answer.
3. Turn the nulls of the keys without details into 0 with `F.coalesce(col, F.lit(0))`, if the task wants 0.

The rule of thumb: never join two "many" tables to the same key before aggregating them.

In [ ]:
# Wrong: two detail tables on the same key multiply each other's rows
wrong = (products.join(orders, "product_id", "left")
                 .join(reviews, "product_id", "left")
                 .groupBy("product_id", "product_name")
                 .agg(F.sum("quantity").alias("items"), F.count("review_id").alias("reviews")))
wrong.orderBy("product_id").show()

+----------+------------+-----+-------+
|product_id|product_name|items|reviews|
+----------+------------+-----+-------+
|       101|      Laptop|    6|      6|
|       102|  Headphones|    3|      2|
|       103|Coffee Maker|    8|      6|
|       104|     Blender|    3|      0|
|       105|       Novel|   24|      6|
|       106|    Cookbook|    3|      2|
|       107|   Desk Lamp| NULL|      0|
+----------+------------+-----+-------+


In [ ]:
# Right: one row per product in each summary, then left joins, then 0 for "no rows"
items = orders.groupBy("product_id").agg(F.sum("quantity").alias("items"))
review_counts = reviews.groupBy("product_id").agg(F.count("*").alias("reviews"))
right = (products.join(items, "product_id", "left")
                 .join(review_counts, "product_id", "left")
                 .select("product_id", "product_name",
                         F.coalesce("items", F.lit(0)).alias("items"),
                         F.coalesce("reviews", F.lit(0)).alias("reviews")))
right.orderBy("product_id").show()

+----------+------------+-----+-------+
|product_id|product_name|items|reviews|
+----------+------------+-----+-------+
|       101|      Laptop|    3|      2|
|       102|  Headphones|    3|      1|
|       103|Coffee Maker|    4|      2|
|       104|     Blender|    3|      0|
|       105|       Novel|   12|      2|
|       106|    Cookbook|    3|      1|
|       107|   Desk Lamp|    0|      0|
+----------+------------+-----+-------+


### Exercise

**Q12 · `etl(customers, orders, reviews)`** For every customer, show name, orders (the number of orders) and reviews (the number of reviews written), both 0 when there are none. Sort by customer_id.

Expected output:

```
+----------------+------+-------+
|            name|orders|reviews|
+----------------+------+-------+
|    Aziz Karimov|     4|      2|
|   Bella Schmidt|     3|      2|
|     Carlos Ruiz|     3|      2|
|Dilnoza Rahimova|     2|      0|
|     Emma Wilson|     2|      1|
|    Farid Aliyev|     1|      1|
|       Grace Lee|     0|      0|
+----------------+------+-------+
```

In [ ]:
def etl(customers, orders, reviews):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders, reviews).show()

## Lesson 4 · Pivot: rows into columns

`df.groupBy(row_key).pivot(column, [values]).agg(expression)` gives one output row per `row_key` and **one column per value** of `column`, holding the aggregate for that combination.

* **Pass the list of values** when you know them. The columns then come out in that order, and a value with no rows still gets its column. Without the list, Spark first runs an extra job to find the values, and sorts them.
* A combination without rows is **null**. Use `.na.fill(0, subset=[the pivot columns])` or `coalesce` if the task wants 0. A plain `.na.fill(0)` fills **every** number column, including a null key column.
* The new columns are **named after the values** (`Books`, `cancelled`). Rename them with `withColumnRenamed` if the output schema wants other names.
* With a single aggregate, the shortcuts `.count()` and `.sum("x")` work after `pivot`.
* The reverse, columns into rows, is `df.unpivot(ids, value_columns, "name_column", "value_column")` (Spark 3.4 and later), or `F.expr("stack(2, 'jan', jan, 'feb', feb)")` on older versions.

On Zillacode, #3 pivots rents by property type and then adds the columns up. [pivot docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html).

In [ ]:
# Without a value list: the values found in the data, sorted; empty combinations are null
(orders.join(products, "product_id")
       .groupBy("customer_id")
       .pivot("category")
       .agg(F.sum("quantity"))
       .orderBy("customer_id")
       .show())

+-----------+-----+-----------+-------+
|customer_id|Books|Electronics|Kitchen|
+-----------+-----+-----------+-------+
|       NULL| NULL|       NULL|      1|
|          1|    3|          2|      1|
|          2|    1|          3|   NULL|
|          3|    4|       NULL|      1|
|          4|    2|       NULL|      2|
|          5|    5|          1|   NULL|
|          6| NULL|       NULL|      2|
+-----------+-----+-----------+-------+


In [ ]:
# With a value list (Home has no orders, but gets a column) and 0 instead of null.
# subset= limits the fill to the pivot columns: a plain na.fill(0) would also turn the guest's null customer_id into 0.
categories = ["Books", "Electronics", "Kitchen", "Home"]
(orders.join(products, "product_id")
       .groupBy("customer_id")
       .pivot("category", categories)
       .agg(F.sum("quantity"))
       .na.fill(0, subset=categories)
       .orderBy("customer_id")
       .show())

+-----------+-----+-----------+-------+----+
|customer_id|Books|Electronics|Kitchen|Home|
+-----------+-----+-----------+-------+----+
|       NULL|    0|          0|      1|   0|
|          1|    3|          2|      1|   0|
|          2|    1|          3|      0|   0|
|          3|    4|          0|      1|   0|
|          4|    2|          0|      2|   0|
|          5|    5|          1|      0|   0|
|          6|    0|          0|      2|   0|
+-----------+-----+-----------+-------+----+


In [ ]:
# The reverse: columns into rows
wide = spark.createDataFrame([(1, 10, 20), (2, 30, None)], "store int, jan int, feb int")
wide.show()
wide.unpivot("store", ["jan", "feb"], "month", "sales").show()

+-----+---+----+
|store|jan| feb|
+-----+---+----+
|    1| 10|  20|
|    2| 30|NULL|
+-----+---+----+

+-----+-----+-----+
|store|month|sales|
+-----+-----+-----+
|    1|  jan|   10|
|    1|  feb|   20|
|    2|  jan|   30|
|    2|  feb| NULL|
+-----+-----+-----+


### Exercises

**Q13 · `etl(orders)`** Make a table with one row per month and one column per status, holding the number of orders, with 0 where there are none. Columns: month, cancelled, delivered, shipped. Sort by month.

Expected output:

```
+-----+---------+---------+-------+
|month|cancelled|delivered|shipped|
+-----+---------+---------+-------+
|    1|        0|        3|      0|
|    2|        1|        2|      1|
|    3|        1|        5|      3|
+-----+---------+---------+-------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q14 · `etl(orders, products)`** For each customer with delivered orders, show the delivered revenue (quantity × price) of each category in its own column: customer_id, Books, Electronics, Kitchen (0 where none), plus `total`. Sort by customer_id. Leave out the guest.

Expected output:

```
+-----------+-----+-----------+-------+------+
|customer_id|Books|Electronics|Kitchen| total|
+-----------+-----+-----------+-------+------+
|          1| 45.0|     1350.0|    0.0|1395.0|
|          2| 25.0|     1500.0|    0.0|1525.0|
|          3| 60.0|        0.0|    0.0|  60.0|
|          5| 75.0|        0.0|    0.0|  75.0|
+-----------+-----+-----------+-------+------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show()

## Lesson 5 · Arrays: `split`, `explode` and `collect_list`

An array column holds a list in every row. You get one from `F.split` (Day 12), or from `F.collect_list` / `F.collect_set` in an `agg`.

| Function | What it does |
|---|---|
| `F.explode(arr)` | one row per item. Rows whose array is null or empty **disappear**. |
| `F.explode_outer(arr)` | the same, but those rows stay, with a null item |
| `F.posexplode(arr)` | also gives each item's position (from 0) |
| `F.size(arr)`, `F.array_contains(arr, v)`, `F.element_at(arr, i)` | count the items, test for one, pick one (from 1) |
| `F.array_distinct`, `F.sort_array`, `F.array_join(arr, ", ")` | remove repeats, sort, turn back into text |
| `F.collect_list(x)`, `F.collect_set(x)` inside `agg` | the reverse of explode: many rows into one array (`set` without repeats) |

The typical job is a column with a list inside a text (`"fast delivery;good price"`): `split` it, `explode` it, then `groupBy` the items to count them.

`collect_list` and `collect_set` keep **no particular order**. When the output must be stable (and a checker compares it), wrap them in `F.sort_array(...)`.

[explode docs](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.explode.html).

In [ ]:
# One row per tag; review 6 (no tags) disappears with explode, stays with explode_outer
reviews.select("review_id", F.explode(F.split("tags", ";")).alias("tag")).show()
reviews.select("review_id", F.explode_outer(F.split("tags", ";")).alias("tag")).filter(F.col("review_id") == 6).show()

+---------+-------------+
|review_id|          tag|
+---------+-------------+
|        1|fast delivery|
|        1|   good price|
|        2|   good price|
|        3|slow delivery|
|        4|       broken|
|        4|slow delivery|
|        5|fast delivery|
|        7|   good price|
|        7|         gift|
|        8|         gift|
+---------+-------------+

+---------+----+
|review_id| tag|
+---------+----+
|        6|NULL|
+---------+----+


In [ ]:
reviews.select(
    "review_id",
    F.split("tags", ";").alias("tag_list"),
    F.array_contains(F.split("tags", ";"), "gift").alias("is_gift"),
).show(truncate=False)

+---------+---------------------------+-------+
|review_id|tag_list                   |is_gift|
+---------+---------------------------+-------+
|1        |[fast delivery, good price]|false  |
|2        |[good price]               |false  |
|3        |[slow delivery]            |false  |
|4        |[broken, slow delivery]    |false  |
|5        |[fast delivery]            |false  |
|6        |NULL                       |NULL   |
|7        |[good price, gift]         |true   |
|8        |[gift]                     |true   |
+---------+---------------------------+-------+


In [ ]:
# Many rows into one array per group, sorted so the result is stable
orders.filter(F.col("customer_id").isNotNull()).groupBy("customer_id").agg(
    F.sort_array(F.collect_list("product_id")).alias("products"),
    F.sort_array(F.collect_set("status")).alias("statuses"),
    F.array_join(F.sort_array(F.collect_set("status")), ", ").alias("statuses_text"),
).orderBy("customer_id").show(truncate=False)

+-----------+--------------------+----------------------+--------------------+
|customer_id|products            |statuses              |statuses_text       |
+-----------+--------------------+----------------------+--------------------+
|1          |[101, 102, 103, 105]|[delivered, shipped]  |delivered, shipped  |
|2          |[101, 102, 106]     |[delivered]           |delivered           |
|3          |[103, 105, 108]     |[cancelled, delivered]|cancelled, delivered|
|4          |[104, 106]          |[cancelled, shipped]  |cancelled, shipped  |
|5          |[101, 105]          |[delivered, shipped]  |delivered, shipped  |
|6          |[103]               |[shipped]             |shipped             |
+-----------+--------------------+----------------------+--------------------+


### Exercises

**Q15 · `etl(reviews)`** Show one row per tag: review_id and tag. Reviews without tags are left out. Sort by review_id, then tag.

Expected output:

```
+---------+-------------+
|review_id|          tag|
+---------+-------------+
|        1|fast delivery|
|        1|   good price|
|        2|   good price|
|        3|slow delivery|
|        4|       broken|
|        4|slow delivery|
|        5|fast delivery|
|        7|         gift|
|        7|   good price|
|        8|         gift|
+---------+-------------+
```

In [ ]:
def etl(reviews):
    # Write code here: return a DataFrame
    return reviews

etl(reviews).show()

**Q16 · `etl(reviews)`** Which tags are used most? Show tag and reviews (the number of reviews with that tag), most first; on a tie, by tag from A to Z.

Expected output:

```
+-------------+-------+
|          tag|reviews|
+-------------+-------+
|   good price|      3|
|fast delivery|      2|
|         gift|      2|
|slow delivery|      2|
|       broken|      1|
+-------------+-------+
```

In [ ]:
def etl(reviews):
    # Write code here: return a DataFrame
    return reviews

etl(reviews).show()

**Q17 · `etl(orders, products)`** For each customer, list the different categories they ordered from, from A to Z, as one text separated by `, `. Show customer_id and categories, sorted by customer_id. Leave out the guest; use an inner join.

Expected output:

```
+-----------+---------------------------+
|customer_id|categories                 |
+-----------+---------------------------+
|1          |Books, Electronics, Kitchen|
|2          |Books, Electronics         |
|3          |Books, Kitchen             |
|4          |Books, Kitchen             |
|5          |Books, Electronics         |
|6          |Kitchen                    |
+-----------+---------------------------+
```

In [ ]:
def etl(orders, products):
    # Write code here: return a DataFrame
    return orders

etl(orders, products).show(truncate=False)

## Mixed problems
Several lessons in one problem, like on Zillacode. About 10 to 15 minutes each.

**Q18 · `etl(orders)`** A monthly report: month (text like `2024-01`), orders, customers (different customers; the guest doesn't count), delivered_items (the quantity of delivered orders) and cancelled_pct (the percent of the month's orders that were cancelled, 1 decimal). Sort by month.

Expected output:

```
+-------+------+---------+---------------+-------------+
|  month|orders|customers|delivered_items|cancelled_pct|
+-------+------+---------+---------------+-------------+
|2024-01|     3|        2|              6|          0.0|
|2024-02|     4|        4|              2|         25.0|
|2024-03|     9|        6|             12|         11.1|
+-------+------+---------+---------------+-------------+
```

In [ ]:
def etl(orders):
    # Write code here: return a DataFrame
    return orders

etl(orders).show()

**Q19 · `etl(reviews, products)`** For each category with reviews, show reviews (the number of reviews), avg_rating (2 decimals) and delivery_mentions (the number of reviews whose tags mention `delivery`). Sort by category.

Expected output:

```
+-----------+-------+----------+-----------------+
|   category|reviews|avg_rating|delivery_mentions|
+-----------+-------+----------+-----------------+
|      Books|      3|      3.33|                0|
|Electronics|      3|      4.33|                3|
|    Kitchen|      2|       2.5|                1|
+-----------+-------+----------+-----------------+
```

In [ ]:
def etl(reviews, products):
    # Write code here: return a DataFrame
    return reviews

etl(reviews, products).show()

**Q20 · `etl(customers, orders)`** How long did each customer take to place a first order? Show customer_id, signup_date, first_order_date and days_to_first_order. Customers without orders show nulls. Sort by customer_id.

Expected output:

```
+-----------+-----------+----------------+-------------------+
|customer_id|signup_date|first_order_date|days_to_first_order|
+-----------+-----------+----------------+-------------------+
|          1| 2024-01-05|      2024-01-10|                  5|
|          2| 2024-01-20|      2024-01-25|                  5|
|          3| 2024-02-02|      2024-02-03|                  1|
|          4| 2024-02-14|      2024-02-15|                  1|
|          5| 2024-03-01|      2024-03-02|                  1|
|          6| 2024-03-10|      2024-03-12|                  2|
|          7| 2024-03-15|            NULL|               NULL|
+-----------+-----------+----------------+-------------------+
```

In [ ]:
def etl(customers, orders):
    # Write code here: return a DataFrame
    return customers

etl(customers, orders).show()

## Answers
Try each exercise yourself first. Click an exercise to open its answer.

<details>
<summary><b>Q1</b> &nbsp;How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered, ...</summary>

```python
def etl(customers):
    return (customers.withColumn("days_registered", F.datediff(F.to_date(F.lit("2024-04-01")), F.col("signup_date")))
                     .select("name", "signup_date", "days_registered")
                     .orderBy("signup_date"))

etl(customers).show()
```

</details>

<details>
<summary><b>Q2</b> &nbsp;For the orders placed in January and February, show order_id, order_date, the last day of that month ...</summary>

```python
def etl(orders):
    return (orders.filter(F.month("order_date") <= 2)
                  .select("order_id", "order_date",
                          F.last_day("order_date").alias("month_end"),
                          F.datediff(F.last_day("order_date"), "order_date").alias("days_left"))
                  .orderBy("order_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q3</b> &nbsp;The sign-up dates come in three formats (2024-01-05, 20/01/2024, 2024/02/02). Turn them into one date column ...</summary>

```python
def etl(signups_raw):
    return (signups_raw.select("signup_id", "signup_date",
                                F.coalesce(F.to_date("signup_date", "yyyy-MM-dd"),
                                           F.to_date("signup_date", "dd/MM/yyyy"),
                                           F.to_date("signup_date", "yyyy/MM/dd")).alias("signup_day"))
                       .orderBy("signup_id"))

etl(signups_raw).show()
```

Each `to_date` gives null when its pattern doesn't fit, and `coalesce` takes the first one that worked.

</details>

<details>
<summary><b>Q4</b> &nbsp;How long does each promotion run, counting both the first and the last day? Show promo_id, start_date, ...</summary>

```python
def etl(promotions):
    return (promotions.select("promo_id", "start_date", "end_date",
                              (F.datediff("end_date", "start_date") + 1).alias("days"))
                      .orderBy("start_date", "promo_id"))

etl(promotions).show()
```

</details>

<details>
<summary><b>Q5</b> &nbsp;Show the evening events (18:00 or later), duplicates removed: customer_id, event_time and page, sorted by ...</summary>

```python
def etl(events):
    return (events.distinct()
                  .filter(F.hour("event_time") >= 18)
                  .orderBy("event_time"))

etl(events).show()
```

</details>

<details>
<summary><b>Q6</b> &nbsp;For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by ...</summary>

```python
def etl(products):
    return (products.groupBy("category")
                    .agg(F.count("*").alias("products"),
                         F.round(F.avg("price"), 2).alias("avg_price"),
                         F.max("price").alias("max_price"))
                    .orderBy("category"))

etl(products).show()
```

</details>

<details>
<summary><b>Q7</b> &nbsp;Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.</summary>

```python
def etl(orders):
    return (orders.groupBy("customer_id")
                  .agg(F.count("*").alias("orders"))
                  .filter(F.col("orders") >= 3)
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q8</b> &nbsp;How many different customers ordered in each month? Show month (1 to 3) and customers. The guest order ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.month("order_date").alias("month"))
                  .agg(F.count_distinct("customer_id").alias("customers"))
                  .orderBy("month"))

etl(orders).show()
```

`count_distinct` skips nulls, so the guest order is left out without a filter.

</details>

<details>
<summary><b>Q9</b> &nbsp;For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show ...</summary>

```python
def etl(orders):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
                       F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q10</b> &nbsp;Which customers order more items per order, on average, than the average order? Show customer_id and ...</summary>

```python
def etl(orders):
    overall = orders.agg(F.avg("quantity")).first()[0]
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.avg("quantity").alias("avg_quantity"))
                  .filter(F.col("avg_quantity") > overall)
                  .select("customer_id", F.round("avg_quantity", 2).alias("avg_quantity"))
                  .orderBy("customer_id"))

etl(orders).show()
```

Filter on the unrounded average, then round for the output: rounding first can move a value across the limit.

</details>

<details>
<summary><b>Q11</b> &nbsp;What share of all items ordered does each category have? Show category, items and share_pct (percent, 1 ...</summary>

```python
def etl(orders, products):
    per_category = (orders.join(products, "product_id")
                          .groupBy("category")
                          .agg(F.sum("quantity").alias("items")))
    total = per_category.agg(F.sum("items")).first()[0]
    return (per_category.withColumn("share_pct", F.round(F.col("items") / total * 100, 1))
                        .orderBy(F.desc("share_pct")))

etl(orders, products).show()
```

</details>

<details>
<summary><b>Q12</b> &nbsp;For every customer, show name, orders (the number of orders) and reviews (the number of reviews written), ...</summary>

```python
def etl(customers, orders, reviews):
    order_counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    review_counts = reviews.groupBy("customer_id").agg(F.count("*").alias("reviews"))
    return (customers.join(order_counts, "customer_id", "left")
                     .join(review_counts, "customer_id", "left")
                     .orderBy("customer_id")
                     .select("name",
                             F.coalesce("orders", F.lit(0)).alias("orders"),
                             F.coalesce("reviews", F.lit(0)).alias("reviews")))

etl(customers, orders, reviews).show()
```

</details>

<details>
<summary><b>Q13</b> &nbsp;Make a table with one row per month and one column per status, holding the number of orders, with 0 where ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.month("order_date").alias("month"))
                  .pivot("status", ["cancelled", "delivered", "shipped"])
                  .count()
                  .na.fill(0)
                  .orderBy("month"))

etl(orders).show()
```

</details>

<details>
<summary><b>Q14</b> &nbsp;For each customer with delivered orders, show the delivered revenue (quantity × price) of each category in ...</summary>

```python
def etl(orders, products):
    wide = (orders.filter((F.col("status") == "delivered") & F.col("customer_id").isNotNull())
                  .join(products, "product_id")
                  .groupBy("customer_id")
                  .pivot("category", ["Books", "Electronics", "Kitchen"])
                  .agg(F.sum(F.col("quantity") * F.col("price")))
                  .na.fill(0))
    return (wide.withColumn("total", F.col("Books") + F.col("Electronics") + F.col("Kitchen"))
                .orderBy("customer_id"))

etl(orders, products).show()
```

Fill the nulls **before** adding the columns: `null + 100` is null.

</details>

<details>
<summary><b>Q15</b> &nbsp;Show one row per tag: review_id and tag. Reviews without tags are left out. Sort by review_id, then tag.</summary>

```python
def etl(reviews):
    return (reviews.select("review_id", F.explode(F.split("tags", ";")).alias("tag"))
                   .orderBy("review_id", "tag"))

etl(reviews).show()
```

</details>

<details>
<summary><b>Q16</b> &nbsp;Which tags are used most? Show tag and reviews (the number of reviews with that tag), most first; on a tie, ...</summary>

```python
def etl(reviews):
    return (reviews.select(F.explode(F.split("tags", ";")).alias("tag"))
                   .groupBy("tag")
                   .agg(F.count("*").alias("reviews"))
                   .orderBy(F.desc("reviews"), "tag"))

etl(reviews).show()
```

</details>

<details>
<summary><b>Q17</b> &nbsp;For each customer, list the different categories they ordered from, from A to Z, as one text separated by , . ...</summary>

```python
def etl(orders, products):
    return (orders.filter(F.col("customer_id").isNotNull())
                  .join(products, "product_id")
                  .groupBy("customer_id")
                  .agg(F.array_join(F.sort_array(F.collect_set("category")), ", ").alias("categories"))
                  .orderBy("customer_id"))

etl(orders, products).show(truncate=False)
```

</details>

<details>
<summary><b>Q18</b> &nbsp;A monthly report: month (text like 2024-01), orders, customers (different customers; the guest doesn't ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.date_format("order_date", "yyyy-MM").alias("month"))
                  .agg(F.count("*").alias("orders"),
                       F.countDistinct("customer_id").alias("customers"),
                       F.sum(F.when(F.col("status") == "delivered", F.col("quantity")).otherwise(0)).alias("delivered_items"),
                       F.round(F.avg(F.when(F.col("status") == "cancelled", 1).otherwise(0)) * 100, 1).alias("cancelled_pct"))
                  .orderBy("month"))

etl(orders).show()
```

The average of a 1/0 column is the share of rows where the condition is true: a handy trick for percentages.

</details>

<details>
<summary><b>Q19</b> &nbsp;For each category with reviews, show reviews (the number of reviews), avg_rating (2 decimals) and ...</summary>

```python
def etl(reviews, products):
    return (reviews.join(products, "product_id")
                   .groupBy("category")
                   .agg(F.count("*").alias("reviews"),
                        F.round(F.avg("rating"), 2).alias("avg_rating"),
                        F.count(F.when(F.col("tags").contains("delivery"), True)).alias("delivery_mentions"))
                   .orderBy("category"))

etl(reviews, products).show()
```

</details>

<details>
<summary><b>Q20</b> &nbsp;How long did each customer take to place a first order? Show customer_id, signup_date, first_order_date and ...</summary>

```python
def etl(customers, orders):
    first_orders = orders.groupBy("customer_id").agg(F.min("order_date").alias("first_order_date"))
    return (customers.join(first_orders, "customer_id", "left")
                     .select("customer_id", "signup_date", "first_order_date",
                             F.datediff("first_order_date", "signup_date").alias("days_to_first_order"))
                     .orderBy("customer_id"))

etl(customers, orders).show()
```

Aggregate first (one row per customer), then a left join: Lesson 3's pattern.

</details>